<a href="https://colab.research.google.com/github/martinthuriaux/Grokking-Universality/blob/main/01_seed_sweep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

OUTPUT_DIR = "/content/drive/MyDrive/grokking_universality/runs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Saving runs to:", OUTPUT_DIR)
print("Folder exists:", os.path.isdir(OUTPUT_DIR))

Saving runs to: /content/drive/MyDrive/grokking_universality/runs
Folder exists: True


Setup + Helper functions

In [3]:
!pip install transformer_lens
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import einops
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ---- Config ----
p = 53              # prime modulus (our choice, != 113)
frac_train = 0.4
seed = 0

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.9 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=e0805dec5291c4343fee346f5de15b73b1acdd951163b7b7ec5edd47d3804893
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator
Device: cuda


In [4]:
def make_dataset(p, seed, frac_train, device):
    # All (a, b) pairs: a_vec runs 0,0,...,0,1,1,...  b_vec runs 0,1,...,p-1,0,1,...
    # Create a vector for a and b of size p^2
    a_vec = einops.repeat(torch.arange(p), "a -> (a b)", b=p)
    b_vec = einops.repeat(torch.arange(p), "b -> (a b)", a=p)
    # A row of length p^2, which is the = token.
    equals = torch.full((p * p,), p)          # the "=" token, index p
    # For each pair, we we find the answer (a + b) mod p
    labels = (a_vec + b_vec) % p

    # inputs: [a, b, =]  shape [p*p, 3]
    inputs = torch.stack([a_vec, b_vec, equals], dim=1).to(device)
    labels = labels.to(device)

    # reproducible train/test split
    torch.manual_seed(seed)
    perm = torch.randperm(p * p)
    n_train = int(frac_train * p * p)
    train_idx, test_idx = perm[:n_train], perm[n_train:]

    return (inputs[train_idx], labels[train_idx],
            inputs[test_idx],  labels[test_idx])

train_x, train_y, test_x, test_y = make_dataset(p, seed, frac_train, device)

print("Train inputs:", train_x.shape, "Train labels:", train_y.shape)
print("Test inputs: ", test_x.shape,  "Test labels: ", test_y.shape)
print("Total pairs: ", p * p)

a_all = einops.repeat(torch.arange(p), "a -> (a b)", b=p)
b_all = einops.repeat(torch.arange(p), "b -> (a b)", a=p)
eq_all = torch.full((p*p,), p)
all_x = torch.stack([a_all, b_all, eq_all], dim=1).to(device)

Train inputs: torch.Size([1123, 3]) Train labels: torch.Size([1123])
Test inputs:  torch.Size([1686, 3]) Test labels:  torch.Size([1686])
Total pairs:  2809


In [5]:
from transformer_lens import HookedTransformer, HookedTransformerConfig

cfg = HookedTransformerConfig(
    n_layers=1,
    d_model=128,
    d_mlp=512,
    n_heads=4,
    d_head=32,
    n_ctx=3,                 # [a, b, =]
    d_vocab=p + 1,           # numbers 0..p-1 plus the "=" token
    act_fn="relu",
    normalization_type=None, # <-- no LayerNorm (the key grokking simplification)
    seed=seed,
)

model = HookedTransformer(cfg).to(device)
print("Total parameters:", sum(par.numel() for par in model.parameters()))

Moving model to device:  cuda
Total parameters: 212022


In [6]:
# cross entropy loss function given the last logit "="
def loss_fn(logits, labels):
    return F.cross_entropy(logits[:, -1, :], labels)

# accuracy function which identifies if the prediction matches the label
def acc_fn(logits, labels):
    preds = logits[:, -1, :].argmax(dim=-1)
    return (preds == labels).float().mean().item()

# if accuracy > 90%, count it as the grokking speed
def grok_epoch(epochs_log, test_accs, thresh=0.9):
    for e, a in zip(epochs_log, test_accs):
        if a >= thresh:
            return e
    return None

In [7]:

def detect_key_frequencies(model, p, threshold_ratio=20):
    """
    Returns the key frequencies of a grokked model, read from its embedding.
    A frequency is 'key' if its power is > threshold_ratio times the noise floor
    (the median power of the non-key bins).
    """
    W_E = model.W_E[:p]                             # number-token embeddings [p, d_model]
    W_E = W_E - W_E.mean(dim=0, keepdim=True)       # drop the constant offset
    fft = torch.fft.fft(W_E, dim=0)                 # FFT down the number axis
    power = (fft.abs() ** 2).sum(dim=1)             # power per frequency [p]

    half = p // 2 + 1
    power = power[:half].clone()                    # unique half
    power[0] = 0.0                                  # ignore frequency 0 (the constant)

    noise_floor = power[power > 0].median()         # the typical "junk" power level
    freqs = torch.arange(half, device=power.device)
    key_freqs = freqs[power > threshold_ratio * noise_floor].tolist()

    return key_freqs, power

Loop

In [8]:
import time, torch, os

N_SEEDS = 100
NUM_EPOCHS = 30000    # safety margin so slow seeds still grok
LOG_EVERY = 200
GROK_ACC_THRESHOLD = 0.9

def train_one_seed(seed):
    # reproducible dataset split + weights for THIS seed
    train_x, train_y, test_x, test_y = make_dataset(p, seed, frac_train, device)

    torch.manual_seed(seed)
    cfg.seed = seed
    model = HookedTransformer(cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3,
                                  weight_decay=1.0, betas=(0.9, 0.98))

    epochs_log, test_accs, test_losses, train_losses = [], [], [], []

    for epoch in range(NUM_EPOCHS):
        model.train()
        optimizer.zero_grad()
        tl = loss_fn(model(train_x), train_y)
        tl.backward()
        optimizer.step()

        if epoch % LOG_EVERY == 0:
            model.eval()
            with torch.no_grad():
                te_logits = model(test_x)
                epochs_log.append(epoch)
                train_losses.append(tl.item())
                test_losses.append(loss_fn(te_logits, test_y).item())
                test_accs.append(acc_fn(te_logits, test_y))

    # final metrics
    model.eval()
    with torch.no_grad():
        final_test_acc = acc_fn(model(test_x), test_y)
    grokked = final_test_acc >= GROK_ACC_THRESHOLD
    ge = grok_epoch(epochs_log, test_accs, thresh=GROK_ACC_THRESHOLD)

    # detect frequencies only if it grokked (junk otherwise)
    key_freqs = detect_key_frequencies(model, p, threshold_ratio=50)[0] if grokked else []

    with torch.no_grad():
      weight_norm = sum((par**2).sum() for par in model.parameters()).sqrt().item()

    # neuron-count per key frequency (how many MLP neurons each frequency recruits)
    neuron_freq_counts = {}
    if grokked:
        with torch.no_grad():
            _, cache = model.run_with_cache(all_x)
            acts = cache["post", 0, "mlp"][:, -1, :]          # [p*p, d_mlp]
        grid = acts.reshape(p, p, -1)
        grid = grid - grid.mean(dim=(0,1), keepdim=True)
        pw = (torch.fft.fft2(grid, dim=(0,1)).abs()**2)
        half = p // 2 + 1
        fp = pw[:half, :half].sum(dim=1) + pw[:half, :half].sum(dim=0)   # [half, d_mlp]
        fp[0] = 0
        dom = fp.argmax(dim=0)                                # each neuron's dominant freq
        import collections
        neuron_freq_counts = dict(collections.Counter(dom.cpu().tolist()))

    return {
        "seed": seed, "p": p, "frac_train": frac_train,
        "state_dict": model.state_dict(),
        "epochs_log": epochs_log,
        "train_losses": train_losses, "test_losses": test_losses, "test_accs": test_accs,
        "final_test_acc": final_test_acc,
        "grokked": bool(grokked),
        "grok_epoch": ge,
        "key_freqs": key_freqs,
        "weight_norm": weight_norm,
        "neuron_freq_counts": neuron_freq_counts,
    }

# ---- run the sweep ----
for seed in range(N_SEEDS):
    path = f"{OUTPUT_DIR}/run_seed{seed:03d}.pt"
    if os.path.exists(path):                    # <-- guard: skip already-done seeds
        print(f"seed {seed:3d} | already done, skipping")
        continue

    t0 = time.time()
    record = train_one_seed(seed)
    torch.save(record, path)
    dt = time.time() - t0
    print(f"seed {seed:3d} | grokked={record['grokked']} "
          f"| acc={record['final_test_acc']:.3f} "
          f"| grok_epoch={record['grok_epoch']} "
          f"| freqs={record['key_freqs']} | {dt:.0f}s | saved")

seed   0 | already done, skipping
seed   1 | already done, skipping
seed   2 | already done, skipping
seed   3 | already done, skipping
seed   4 | already done, skipping
seed   5 | already done, skipping
seed   6 | already done, skipping
seed   7 | already done, skipping
seed   8 | already done, skipping
seed   9 | already done, skipping
seed  10 | already done, skipping
seed  11 | already done, skipping
seed  12 | already done, skipping
seed  13 | already done, skipping
seed  14 | already done, skipping
seed  15 | already done, skipping
seed  16 | already done, skipping
seed  17 | already done, skipping
seed  18 | already done, skipping
seed  19 | already done, skipping
seed  20 | already done, skipping
seed  21 | already done, skipping
seed  22 | already done, skipping
seed  23 | already done, skipping
seed  24 | already done, skipping
seed  25 | already done, skipping
seed  26 | already done, skipping
seed  27 | already done, skipping
seed  28 | already done, skipping
seed  29 | alr

To fix errors with the key frequency detector, I created a new one which uses the fraction power(freq) / power(all freq) as a key criteria.

In [17]:
def detect_key_frequencies_v2(state_dict_or_model, p, power_frac=0.07):
    """A frequency is 'key' if it carries more than power_frac of total power."""
    if isinstance(state_dict_or_model, dict):
        W_E = state_dict_or_model["embed.W_E"][:p].float()
    else:
        W_E = state_dict_or_model.W_E[:p]
    W_E = W_E - W_E.mean(dim=0, keepdim=True)
    power = (torch.fft.fft(W_E, dim=0).abs()**2).sum(dim=1)
    half = p // 2 + 1
    power = power[:half].clone()
    power[0] = 0
    frac = power / power.sum()
    key_freqs = torch.arange(half)[(frac > power_frac).cpu()].tolist()
    return key_freqs, frac

Test to see if it works

In [18]:
import torch, glob

expected = {
    0: [6, 8, 19],
    1: [3, 7, 26],
    2: [1, 20, 25],
    3: [5, 9, 13, 15],
    24: None,   # we know it's 4 spikes ~[5,13,16,20]; confirm by eye
}

for seed in [0, 1, 2, 3, 24]:
    d = torch.load(f"{OUTPUT_DIR}/run_seed{seed:03d}.pt", map_location="cpu")
    kf, frac = detect_key_frequencies_v2(d["state_dict"], p)
    print(f"seed {seed:3d}: detected {kf}"
          + (f"   (expected {expected[seed]})" if expected[seed] else "   <- the failure case"))

seed   0: detected [6, 8, 16, 19]   (expected [6, 8, 19])
seed   1: detected [3, 7, 26]   (expected [3, 7, 26])
seed   2: detected [1, 20, 25]   (expected [1, 20, 25])
seed   3: detected [5, 9, 13, 15]   (expected [5, 9, 13, 15])
seed  24: detected [5, 13, 16, 20]   <- the failure case


Sweep to identify what power_frac is best, and if it sits in the middle of the correct range or not.

In [19]:
import collections

files = sorted(glob.glob(f"{OUTPUT_DIR}/run_seed*.pt"))
for pf in [0.04, 0.05, 0.07, 0.10, 0.12]:
    counts = collections.Counter()
    n_empty = 0
    for f in files:
        d = torch.load(f, map_location="cpu")
        if not d["grokked"]:
            continue
        kf, _ = detect_key_frequencies_v2(d["state_dict"], p, power_frac=pf)
        counts[len(kf)] += 1
        n_empty += (len(kf) == 0)
    print(f"power_frac={pf}: count distribution {dict(sorted(counts.items()))}, empty={n_empty}")

power_frac=0.04: count distribution {2: 1, 3: 51, 4: 45, 5: 3}, empty=0
power_frac=0.05: count distribution {2: 1, 3: 51, 4: 46, 5: 2}, empty=0
power_frac=0.07: count distribution {2: 2, 3: 52, 4: 45, 5: 1}, empty=0
power_frac=0.1: count distribution {2: 3, 3: 63, 4: 33, 5: 1}, empty=0
power_frac=0.12: count distribution {2: 5, 3: 69, 4: 25, 5: 1}, empty=0


In [20]:
for f in files:
    d = torch.load(f, map_location="cpu")
    if d["grokked"]:
        kf, _ = detect_key_frequencies_v2(d["state_dict"], p, power_frac=0.07)
        d["key_freqs_v1"] = d["key_freqs"]      # keep the old detection for provenance
        d["key_freqs"] = kf
        torch.save(d, f)
print("All records updated.")

All records updated.
